# Conditional Random Fields (CRF) for Information Extraction

## Overview

This notebook presents a **supervised sequence labeling approach using Conditional Random Fields (CRF)** for structured information extraction from clinical trial abstracts in the **EBM-NLP dataset**. The goal is to identify and extract key elements of the **PIO framework** Population, Intervention, and Outcome—from unstructured biomedical text.

Unlike rule-based methods, CRF models learn patterns directly from annotated data by leveraging contextual and linguistic features across tokens in a sentence. This makes them well-suited for sequence labeling tasks where dependencies between neighboring words are important.

In this implementation, the model is trained using engineered features such as:

- Token-level features (word identity, shape, case information)
- N-gram context features (previous and next tokens)


In [ ]:
import pandas as pd
import numpy as np
import ast

In [90]:
train_dataset = df_train.iloc[300:1300, : ]
test_dataset = df_train.iloc[:200, :]

In [91]:
tokens = train_dataset["Tokens"].apply(ast.literal_eval)
ner_bio_tags = train_dataset["Merged BIO Labels"].apply(ast.literal_eval)

In [108]:
test_tokens = test_dataset["Tokens"].apply(ast.literal_eval)
test_bio_tags = test_dataset["Merged BIO Labels"].apply(ast.literal_eval)

In [120]:
train_crf = [list(zip(sentence_tokens, sentence_tags))for sentence_tokens, sentence_tags in zip(tokens, ner_bio_tags)]
test_crf = [ i for i in test_tokens]
true_labels = [ y for y in test_bio_tags]

In [94]:
import nltk
def train_CRF_NER_tagger(train_set):
    tagger = nltk.tag.CRFTagger()
    tagger.train(train_set, 'model.crf.tagger')
    return tagger  # return the trained model
tagger = train_CRF_NER_tagger(train_crf) 

In [98]:
predicted_tags = tagger.tag_sents(test_crf)

In [113]:
predicted_tags[1]

[('Xylitol', 'B-INT'),
 ('for', 'O'),
 ('prevention', 'O'),
 ('of', 'O'),
 ('acute', 'B-PART'),
 ('otitis', 'I-PART'),
 ('media', 'I-PART'),
 ('.', 'I-PART')]

In [104]:
pred_labels = [[k[1] for k in i ] for i in predicted_tags]

In [121]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         INT       0.42      0.20      0.27      1561
         OUT       0.25      0.09      0.13      1692
        PART       0.07      0.01      0.02       687

   micro avg       0.32      0.12      0.17      3940
   macro avg       0.25      0.10      0.14      3940
weighted avg       0.29      0.12      0.17      3940



In [122]:
import re, unicodedata

class CustomCRFTagger(nltk.tag.CRFTagger):
    _current_tokens = None
    
    def _get_features(self, tokens, idx):
            """
            Extract basic features about this word including
                - Current word
                - is it capitalized?
                - Does it have punctuation?
                - Does it have a number?
                - Suffixes up to length 3

            Note that : we might include feature over previous word, next word etc.

            :return: a list which contains the features
            :rtype: list(str)
            """
            token = tokens[idx]

            feature_list = []

            if not token:
                return feature_list

            # Capitalization
            if token[0].isupper():
                feature_list.append("CAPITALIZATION")

            # Number
            if re.search(self._pattern, token) is not None:
                feature_list.append("HAS_NUM")

            # Punctuation
            punc_cat = {"Pc", "Pd", "Ps", "Pe", "Pi", "Pf", "Po"}
            if all(unicodedata.category(x) in punc_cat for x in token):
                feature_list.append("PUNCTUATION")

            # Suffix up to length 3
            if len(token) > 1:
                feature_list.append("SUF_" + token[-1:])
            if len(token) > 2:
                feature_list.append("SUF_" + token[-2:])
            if len(token) > 3:
                feature_list.append("SUF_" + token[-3:])

                
            # Current word
            feature_list.append("WORD_" + token)
            
            if idx == 0:
                feature_list.append("BOS")
                
            if idx > 0:
                feature_list.append("PREV_" + tokens[idx-1])
                
            if  idx < (len(tokens)-1):
                feature_list.append("NEXT_" + tokens[idx + 1])
                
            if idx == (len(tokens)-1):
                feature_list.append("EOS")
                
            feature_list.append("LEN_"+ str(len(token)))
            
            return feature_list

In [123]:
import nltk
def train_CRF_NER_tagger(train_set):
    tagger = CustomCRFTagger()
    tagger.train(train_set, 'model.crf.tagger')
    return tagger  # return the trained model
tagger = train_CRF_NER_tagger(train_crf)

In [124]:
predicted_tags = tagger.tag_sents(test_crf)

In [126]:
pred_labels = [[k[1] for k in i ] for i in predicted_tags]

In [127]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         INT       0.47      0.29      0.36      1561
         OUT       0.38      0.19      0.25      1692
        PART       0.30      0.11      0.16       687

   micro avg       0.41      0.21      0.28      3940
   macro avg       0.38      0.20      0.26      3940
weighted avg       0.40      0.21      0.28      3940

